# 04 — Window inspection

Per `plan.md` §11 step 4 + §12. Sweep the catalog and confirm:

1. The `64×64` window covers ≥ 95% of B11 holes' angular sizes at the
   cube's native pixel scale (§2.1 + §12 first bullet).
2. Augmentations look physically plausible and do not destroy the
   shell signature (eyeball check on §2.3 grid).
3. Negative-sample windows are clearly distinct from positives.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from hishells.augment import AugmentConfig, Augmenter
from hishells.catalog import LOGO_GALAXIES_19, load_catalog
from hishells.cubes import channel_width_kms, load_cube, sigma_rms
from hishells.pvcut import extract_window_for_hole, window_extent_for_hole
from hishells.windows import NegSampleConfig, normalize_window, sample_negatives

In [ ]:
REPO = Path('..').resolve()
DATA = REPO / 'Data' / 'THINGS'
cat = load_catalog(REPO / 'Data' / 'J_AJ_141_23')
downloaded = sorted(p.name.replace('_NA_CUBE_THINGS.FITS', '')
                    for p in DATA.glob('*_NA_CUBE_THINGS.FITS'))
print(f'{len(downloaded)} cubes downloaded')

## 1. Window-extent coverage

For each downloaded galaxy, compute the per-hole pos-extent in pixels
and confirm that 64 px is enough at the chosen `pos_factor=2.0`.

In [ ]:
rows = []
for gid in downloaded:
    cube = load_cube(DATA / f'{gid}_NA_CUBE_THINGS.FITS')
    holes = cat.holes[(cat.holes['galaxy_id'] == gid) & (cat.holes['hole_type'].isin([2, 3]))]
    for _, h in holes.iterrows():
        ext = window_extent_for_hole(
            diameter_arcsec=h['diameter_arcsec'],
            vexp_kms=h['vexp_kms'],
            sigma_gas_kms=h['sigma_gas_kms'],
        )
        rows.append({
            'galaxy_id': gid,
            'pos_extent_pix': 2 * ext.pos_extent_arcsec / cube.pixel_scale_arcsec,
            'vel_extent_kms': 2 * ext.vel_extent_kms,
            'channel_width_kms': channel_width_kms(cube),
        })
import pandas as pd
ext_df = pd.DataFrame(rows)
print(f'{(ext_df["pos_extent_pix"] <= 64).mean():.1%} of holes covered by 64 px windows')
ext_df['pos_extent_pix'].hist(bins=40)
plt.axvline(64, color='red', label='64 px')
plt.xlabel('pos extent (px) at native pixel scale')
plt.legend()

If the coverage above is < 95%, either bump `window_pix` to 96 or
drop the largest-diameter holes from training (§2.4 amendment).

## 2. Augmentation grid

Take one B11 hole, render the raw window, then apply each augmentation
in turn so we can see what the model actually trains on.

In [ ]:
GALAXY = downloaded[0]
cube = load_cube(DATA / f'{GALAXY}_NA_CUBE_THINGS.FITS')
sigma = sigma_rms(cube)
ch = channel_width_kms(cube)
pos_holes = cat.holes[(cat.holes['galaxy_id'] == GALAXY) & (cat.holes['hole_type'].isin([2, 3]))]
if len(pos_holes) == 0:
    raise RuntimeError(f'No type-2/3 holes for {GALAXY}; try a different downloaded galaxy.')
h = pos_holes.iloc[0]
raw = normalize_window(extract_window_for_hole(cube, h.to_dict(), window_pix=64), sigma)

configs = {
    'raw': AugmentConfig(noise_enabled=False, pos_roll_enabled=False, vel_roll_enabled=False, flip_pos_enabled=False, flip_vel_enabled=False, zoom_enabled=False),
    'noise only': AugmentConfig(noise_enabled=True, pos_roll_enabled=False, vel_roll_enabled=False, flip_pos_enabled=False, flip_vel_enabled=False, zoom_enabled=False),
    'flip pos': AugmentConfig(flip_pos_p=1.0, noise_enabled=False, pos_roll_enabled=False, vel_roll_enabled=False, flip_vel_enabled=False, zoom_enabled=False),
    'flip vel': AugmentConfig(flip_vel_p=1.0, noise_enabled=False, pos_roll_enabled=False, vel_roll_enabled=False, flip_pos_enabled=False, zoom_enabled=False),
    'zoom 0.85': AugmentConfig(zoom_low=0.85, zoom_high=0.85, noise_enabled=False, pos_roll_enabled=False, vel_roll_enabled=False, flip_pos_enabled=False, flip_vel_enabled=False),
    'zoom 1.15': AugmentConfig(zoom_low=1.15, zoom_high=1.15, noise_enabled=False, pos_roll_enabled=False, vel_roll_enabled=False, flip_pos_enabled=False, flip_vel_enabled=False),
    'all default': AugmentConfig(),
    'all + beam': AugmentConfig(beam_perturb_enabled=True),
}
fig, axes = plt.subplots(1, len(configs), figsize=(2.2 * len(configs), 2.5))
for ax, (name, cfg) in zip(axes, configs.items()):
    rng = np.random.default_rng(0)
    aug = Augmenter(cfg)
    out = aug(raw, rng, channel_width_kms=ch)
    ax.imshow(out, origin='lower', cmap='magma', vmin=-2, vmax=8)
    ax.set_title(name, fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()

## 3. Positives vs negatives

Pull a small batch of positives and matched negatives from the same
cube, normalize, and render side-by-side. Confirms the negative
sampler isn't accidentally producing shell-looking windows.

In [ ]:
negs = sample_negatives(cube, pos_holes, NegSampleConfig(ratio=2.0, rng_seed=0), cube_sigma=sigma)
fig, axes = plt.subplots(2, 6, figsize=(13, 4.5))
for ax, (_, h) in zip(axes[0], pos_holes.head(6).iterrows()):
    win = normalize_window(extract_window_for_hole(cube, h.to_dict(), window_pix=64), sigma)
    ax.imshow(win, origin='lower', cmap='magma', vmin=-2, vmax=8)
    ax.set_title(f'POS hole {h.hole_idx}', fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
for ax, (_, n) in zip(axes[1], negs.head(6).iterrows()):
    win = normalize_window(extract_window_for_hole(cube, n.to_dict(), window_pix=64), sigma)
    ax.imshow(win, origin='lower', cmap='magma', vmin=-2, vmax=8)
    ax.set_title(f'NEG ({n.neg_kind})', fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()

## Verification table to log in `plan.md`

Once the three checks above pass, record the resolved values for the
`[default — verify]` items in §2.1:

| Parameter | Verified value |
|---|---|
| `window_pix` | 64 |
| `pos_extent` factor | 2.0 |
| `vel_extent` factor / floor | 2.0 / 20 km/s |
| Coverage of holes by 64 px | (fill in from histogram) |